
---
date: "2026-10-11"
date-modified: last-modified
format:
  html:
    toc: true
---



# Eigenvalues and Eigenvectors

**Prerequisites:** [Kernel and Image](kernel-image-and-rank-nullity.ipynb) · [The Determinant](determinant.ipynb) · [Trace](trace.ipynb) · [Direct Sums](direct-sums-and-complements.ipynb)

Most vectors are turned to a new direction by a linear operator. An **eigenvector** is a vector whose direction is kept: the operator only stretches, shrinks or flips it, by a factor called the **eigenvalue**. Eigenvectors are the natural axes of an operator, and finding them is the first step towards choosing a basis in which the operator's matrix is as simple as possible.

Throughout, $V$ is a [vector space](vector-spaces.ipynb) of dimension $n$ over a field $K$, $f : V \to V$ is a [linear operator](linear-maps.ipynb#def-linear-map), and $A \in M_n(K)$ is a square matrix, regarded as the operator $x \mapsto A x$ on $K^n$.

## 1\. Eigenvalues, Eigenvectors, and Eigenspaces

::: {#def-eigenvalue-eigenvector}
## Eigenvalue, Eigenvector, Eigenspace

1. $\lambda \in K$ is an **eigenvalue** of $f$ if $f(v) = \lambda v$ for some **non-zero** $v \in V$.
2. Every non-zero $v$ with $f(v) = \lambda v$ is an **eigenvector** for $\lambda$.
3. The **eigenspace** of $\lambda$ is $E_\lambda = \ker(f - \lambda\,\mathrm{id}_V) = \{v \in V \mid f(v) = \lambda v\}$, a [subspace](kernel-image-and-rank-nullity.ipynb#def-kernel-image) consisting of the eigenvectors for $\lambda$ together with $0$.

For a matrix $A$, these are the eigenvalues, eigenvectors and eigenspaces of $x \mapsto A x$.
:::

::: {.callout-warning}
## The Zero Vector Is Never an Eigenvector
$f(0) = \lambda \cdot 0$ holds for every $\lambda$, so allowing $v = 0$ would make every scalar an eigenvalue. The scalar $\lambda = 0$, on the other hand, may well be an eigenvalue: this happens exactly when $\ker f \ne \{0\}$, i.e. when $f$ is not [invertible](invertible-linear-maps.ipynb#cor-equal-dim-invertibility).
:::

::: {#exm-eigen}
## Eigenvectors in Practice

1. **A symmetric matrix.** For $A = \begin{pmatrix} 1 & 2 \\ 2 & 1 \end{pmatrix}$: $A(1, 1)^{\mathsf T} = (3, 3)^{\mathsf T}$ and $A(1, -1)^{\mathsf T} = (-1, 1)^{\mathsf T}$. So $(1, 1)$ is an eigenvector with eigenvalue $3$ and $(1, -1)$ one with eigenvalue $-1$.
2. **A rotation.** Rotating $\mathbb{R}^2$ by $90^\circ$ keeps no line through the origin fixed, so it has no real eigenvalues.
3. **Differentiation.** On the space of differentiable functions $\mathbb{R} \to \mathbb{R}$, $\frac{d}{dt} e^{\lambda t} = \lambda e^{\lambda t}$: every real $\lambda$ is an eigenvalue of $\frac{d}{dt}$, with eigenvector $e^{\lambda t}$. (This space is infinite-dimensional.)
4. **A projection.** The projection of $\mathbb{R}^3$ onto the $xy$-plane along the $z$-axis has the eigenvalues $1$ and $0$: $E_1$ is the $xy$-plane and $E_0$ is the $z$-axis.
:::

The animation applies the matrix of example 1. Sixteen vectors around the unit circle are carried along with the grid. Almost all of them are turned off their original line. Only the vectors on the two dashed lines stay on them: those along $(1, 1)$ are stretched by $3$, those along $(1, -1)$ are flipped ($\lambda = -1$). The unit circle becomes an ellipse whose axes are the eigenvector lines.


In [1]:

from la_manim import *

class EigenvectorsKeepTheirLine(GridScene):
    def construct(self):
        self.plane.set_stroke(opacity=0.35)
        for d, col in (([1, 1, 0], C_YELLOW), ([1, -1, 0], C_AQUA)):
            self.add(DashedLine(-6 * np.array(d), 6 * np.array(d), color=col, stroke_width=2, dash_length=0.12))
        circle = self.track(Circle(radius=1, color=C_INK, stroke_width=2))
        self.add(circle)
        for k in range(16):
            t = 2 * np.pi * k / 16 + np.pi / 4
            eig = k % 4 == 0                          # angles 45, 135, 225, 315 degrees
            col = (C_YELLOW if k % 8 == 0 else C_AQUA) if eig else C_MUTED
            self.vec([np.cos(t), np.sin(t)], col)
        cap = caption(r"A = \begin{pmatrix} 1 & 2 \\ 2 & 1 \end{pmatrix}")
        self.play(FadeIn(cap)); self.wait(1)
        self.transform([[1, 2], [2, 1]], run_time=4)
        lab = VGroup(MathTex(r"A v = 3v", color=C_YELLOW), MathTex(r"A v = -v", color=C_AQUA)
                     ).arrange(DOWN, aligned_edge=LEFT).scale(0.75).to_corner(DR, buff=0.3)
        self.play(FadeIn(BackgroundRectangle(lab, fill_opacity=0.85, buff=0.12, color=config.background_color)), FadeIn(lab))
        msg = text_caption(r"Only eigenvectors stay on their own line")
        self.play(FadeIn(msg)); self.wait(2.5)

show(EigenvectorsKeepTheirLine)



## 2\. The Characteristic Polynomial

::: {#thm-characteristic-equation}
## Eigenvalues Are the Roots of the Characteristic Polynomial

$\lambda \in K$ is an eigenvalue of $A$ if and only if
$$p_A(\lambda) := \det(\lambda I_n - A) = 0.$$
$p_A$ is the **characteristic polynomial** of $A$ and $p_A(\lambda) = 0$ its **characteristic equation**. Similar matrices have the same characteristic polynomial, so $p_f := p_{[f]_{\mathcal{B}}}$ does not depend on the basis $\mathcal{B}$, and the eigenvalues of $f$ are the roots of $p_f$ in $K$.
:::

::: {.callout-tip collapse="true"}
## Proof of @thm-characteristic-equation
$\lambda$ is an eigenvalue $\iff$ $(\lambda I_n - A) v = 0$ for some $v \ne 0$ $\iff$ $\ker(\lambda I_n - A) \ne \{0\}$ $\iff$ $\lambda I_n - A$ is not invertible ([equal dimensions](kernel-image-and-rank-nullity.ipynb#cor-dimension-constraints)) $\iff$ $\det(\lambda I_n - A) = 0$ ([invertibility criterion](determinant.ipynb#thm-det-invertibility)).

If $B = P^{-1} A P$, then $\lambda I_n - B = P^{-1}(\lambda I_n - A) P$, so $\det(\lambda I_n - B) = \det(\lambda I_n - A)$ by [multiplicativity](determinant.ipynb#thm-det-multiplicative). Since the matrices of $f$ in different bases are [similar](change-of-basis-and-similarity.ipynb#thm-operator-change-of-basis), $p_f$ is well defined. $\blacksquare$
:::

::: {#prp-char-poly-shape}
## Shape of the Characteristic Polynomial

$p_A$ is a polynomial in $\lambda$ of degree $n$ with leading coefficient $1$ (**monic**), and
$$p_A(\lambda) = \lambda^n - (\operatorname{tr} A)\,\lambda^{n-1} + \dots + (-1)^n \det A.$$
:::

::: {.callout-tip collapse="true"}
## Proof of @prp-char-poly-shape
In the [Leibniz formula](determinant.ipynb#thm-det-existence) for $\det(\lambda I_n - A)$, each term is a product of entries $\lambda\delta_{ij} - a_{ij}$, so it is a polynomial in $\lambda$ of degree at most the number of diagonal factors. A permutation $\sigma \ne \mathrm{id}$ moves at least two indices, so its term has degree at most $n - 2$. The identity contributes $\prod_i (\lambda - a_{ii}) = \lambda^n - (a_{11} + \dots + a_{nn})\lambda^{n-1} + \dots$. Hence $p_A$ is monic of degree $n$ with $\lambda^{n-1}$-coefficient $-\operatorname{tr} A$. The constant term is $p_A(0) = \det(-A) = (-1)^n \det A$. $\blacksquare$
:::

::: {#exm-char-poly}
## Computing Eigenvalues

For $A = \begin{pmatrix} 1 & 2 \\ 2 & 1 \end{pmatrix}$: $p_A(\lambda) = (\lambda - 1)^2 - 4 = \lambda^2 - 2\lambda - 3 = (\lambda - 3)(\lambda + 1)$. The eigenvalues are $3$ and $-1$; solving $(A - 3I)v = 0$ and $(A + I)v = 0$ gives $E_3 = \operatorname{span}(1, 1)$ and $E_{-1} = \operatorname{span}(1, -1)$. In agreement with the proposition, $\operatorname{tr} A = 2 = 3 + (-1)$ and $\det A = -3 = 3 \cdot (-1)$.
:::

## 3\. Algebraically Closed Fields

Whether an operator has eigenvalues at all depends on the field, since $p_A$ may have no roots in $K$. Several results below, and the [Triangularization Theorem](triangularization-theorem.ipynb), assume that every polynomial has roots.

::: {#def-algebraically-closed-field}
## Algebraically Closed Field

A field $K$ is **algebraically closed** if every polynomial of degree $n \ge 1$ with coefficients in $K$ factors into linear factors over $K$:
$$a_n x^n + \dots + a_0 = a_n (x - \mu_1) \cdots (x - \mu_n), \qquad \mu_i \in K.$$
Equivalently, every non-constant polynomial over $K$ has a root in $K$.
:::

- $\mathbb{C}$ is algebraically closed: this is the **Fundamental Theorem of Algebra**.
- $\mathbb{R}$ is not: $x^2 + 1$ has no real root.

::: {.callout-warning}
## Over $\mathbb{R}$ an Operator May Have No Eigenvalue
The rotation $R = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}$ has $p_R(\lambda) = \lambda^2 + 1$, with no real roots, so it has no real eigenvalues. Over $\mathbb{C}$ it has the eigenvalues $\pm i$, with eigenvectors $(1, \mp i)$. Over an algebraically closed field every operator on a space $V \ne \{0\}$ has at least one eigenvalue, because $p_f$ has degree $n \ge 1$.
:::

## 4\. Eigenvectors for Distinct Eigenvalues Are Independent

::: {#thm-distinct-eigenvalues-independent}
## Independence of Eigenvectors

If $v_1, \dots, v_k$ are eigenvectors of $f$ for **pairwise distinct** eigenvalues $\lambda_1, \dots, \lambda_k$, then $v_1, \dots, v_k$ are [linearly independent](linear-independence.ipynb#def-linear-independence).
:::

::: {.callout-tip collapse="true"}
## Proof of @thm-distinct-eigenvalues-independent
Suppose not, and among all non-trivial relations between the $v_i$ choose one with the fewest non-zero coefficients; after renumbering,
$$c_1 v_1 + \dots + c_l v_l = 0, \qquad \text{all } c_i \ne 0. \qquad (\ast)$$
Since $v_1 \ne 0$, $l \ge 2$. Apply $f$ to $(\ast)$, and separately multiply $(\ast)$ by $\lambda_l$:
$$c_1 \lambda_1 v_1 + \dots + c_l \lambda_l v_l = 0, \qquad c_1 \lambda_l v_1 + \dots + c_l \lambda_l v_l = 0.$$
Subtracting, $c_1(\lambda_1 - \lambda_l) v_1 + \dots + c_{l-1}(\lambda_{l-1} - \lambda_l) v_{l-1} = 0$. All these coefficients are non-zero, because $c_i \ne 0$ and $\lambda_i \ne \lambda_l$. This relation is non-trivial and shorter than $(\ast)$, a contradiction. $\blacksquare$
:::

::: {#cor-direct-sum-eigenspaces}
## Eigenspaces Form a Direct Sum

For pairwise distinct eigenvalues $\lambda_1, \dots, \lambda_k$, the sum $E_{\lambda_1} + \dots + E_{\lambda_k}$ is [direct](direct-sums-and-complements.ipynb#def-direct-sum-multiple). In particular $f$ has at most $n$ distinct eigenvalues.
:::

::: {.callout-tip collapse="true"}
## Proof of @cor-direct-sum-eigenspaces
By the [criterion for direct sums](direct-sums-and-complements.ipynb#prp-direct-sum-criterion) it suffices to show that $u_1 + \dots + u_k = 0$ with $u_i \in E_{\lambda_i}$ forces all $u_i = 0$. The non-zero $u_i$ are eigenvectors for distinct eigenvalues, hence independent by @thm-distinct-eigenvalues-independent, and a sum of independent vectors with coefficients $1$ cannot be $0$. So no $u_i$ is non-zero. Choosing one eigenvector per eigenvalue gives an independent set, which has at most $n$ elements. $\blacksquare$
:::

## 5\. Trace and Determinant from the Eigenvalues

::: {#thm-characteristic-coefficients}
## Spectral Formulas for Trace and Determinant

Suppose $p_A$ splits into linear factors over $K$ (always the case if $K$ is algebraically closed): $p_A(\lambda) = (\lambda - \lambda_1) \cdots (\lambda - \lambda_n)$, the eigenvalues listed with multiplicity. Then
$$\operatorname{tr} A = \lambda_1 + \dots + \lambda_n, \qquad \det A = \lambda_1 \lambda_2 \cdots \lambda_n.$$
:::

::: {.callout-tip collapse="true"}
## Proof of @thm-characteristic-coefficients
Multiplying out, $(\lambda - \lambda_1)\cdots(\lambda - \lambda_n) = \lambda^n - (\lambda_1 + \dots + \lambda_n)\lambda^{n-1} + \dots + (-1)^n \lambda_1 \cdots \lambda_n$. Compare with @prp-char-poly-shape. $\blacksquare$
:::

::: {#cor-intermediate-invariants}
## All Coefficients Are Sums of Principal Minors

Write $p_A(\lambda) = \sum_{k=0}^n (-1)^k s_k(A)\,\lambda^{n-k}$. Then $s_k(A)$ is the sum of the $\binom{n}{k}$ **principal $k \times k$ minors** of $A$ (determinants of the submatrices with the same set of $k$ rows and columns). If $p_A$ splits, $s_k(A)$ equals the $k$-th elementary symmetric polynomial $\sum_{i_1 < \dots < i_k} \lambda_{i_1} \cdots \lambda_{i_k}$ of the eigenvalues. Each $s_k$ is a similarity invariant; $s_1 = \operatorname{tr}$ and $s_n = \det$.
:::

::: {.callout-tip collapse="true"}
## Proof of @cor-intermediate-invariants
The $j$-th column of $\lambda I_n - A$ is $\lambda e_j - A_j$ ($A_j$ the $j$-th column of $A$). Expanding $\det$ by linearity in every column, choose for each $j$ either $\lambda e_j$ or $-A_j$. For the set $S$ of columns where $-A_j$ is chosen, the term is $\lambda^{n-|S|}\,(-1)^{|S|}\det M_S$, where $M_S$ has column $A_j$ for $j \in S$ and $e_j$ for $j \notin S$. Expanding $\det M_S$ along the unit columns ([Laplace](laplace-expansion-adjugate-and-cramers-rule.ipynb#lem-unit-column)) deletes those rows and columns, each with sign $+1$ since the unit vector $e_j$ sits on the diagonal, and leaves the principal minor of $A$ on the index set $S$. Collecting the terms with $|S| = k$ gives the claim. The statement about eigenvalues follows by expanding $\prod (\lambda - \lambda_i)$, and invariance from @thm-characteristic-equation. $\blacksquare$
:::

## 6\. Polynomials of an Operator

For a polynomial $p(x) = c_0 + c_1 x + \dots + c_m x^m \in K[x]$ define $p(A) = c_0 I_n + c_1 A + \dots + c_m A^m$, and similarly $p(f) = c_0\,\mathrm{id} + c_1 f + \dots + c_m f^m$ with $f^k = f \circ \dots \circ f$.

::: {#thm-spectral-mapping-polynomial}
## Eigenvectors of $p(A)$

If $A v = \lambda v$, then $A^k v = \lambda^k v$ for all $k \ge 0$ and $p(A) v = p(\lambda) v$ for every polynomial $p$. So each eigenvector of $A$ is an eigenvector of $p(A)$, with eigenvalue $p(\lambda)$.
:::

::: {.callout-tip collapse="true"}
## Proof of @thm-spectral-mapping-polynomial
$A^0 v = v$, and if $A^k v = \lambda^k v$, then $A^{k+1} v = A(\lambda^k v) = \lambda^k A v = \lambda^{k+1} v$. Hence $p(A) v = \sum_k c_k A^k v = \sum_k c_k \lambda^k v = p(\lambda) v$. $\blacksquare$
:::

::: {.callout-warning}
## Eigenvalues Do Not Simply Add
The theorem works because all powers of $A$ share the eigenvector $v$. For two different matrices there is no such relation: if $\lambda$ is an eigenvalue of $A$ and $\mu$ one of $B$, then $\lambda + \mu$ is in general **not** an eigenvalue of $A + B$, because $A$ and $B$ need not have a common eigenvector.
:::

The code checks the statements of this note on a $4 \times 4$ matrix built to have eigenvalues $4, 2.5, -1, 0.5$.


In [2]:

import numpy as np

rng = np.random.default_rng(101)
target = np.array([4.0, 2.5, -1.0, 0.5])
P = rng.normal(size=(4, 4))
A = P @ np.diag(target) @ np.linalg.inv(P)

evals, evecs = np.linalg.eig(A)
order = np.argsort(-evals.real); evals, evecs = evals[order].real, evecs[:, order].real
print("eigenvalues:", np.round(evals, 6))
print("A v = lambda v for all four:", all(np.allclose(A @ evecs[:, i], evals[i] * evecs[:, i]) for i in range(4)))
print("eigenvectors independent (det of the matrix of eigenvectors):", round(np.linalg.det(evecs), 4))

c = np.poly(A)                                        # coefficients of det(lambda I - A)
print("char. poly coefficients:", np.round(c, 6))
print("tr A =", round(np.trace(A), 6), "= sum of eigenvalues", round(evals.sum(), 6))
print("det A =", round(np.linalg.det(A), 6), "= product of eigenvalues", round(evals.prod(), 6))
s2 = sum(np.linalg.det(A[np.ix_([i, j], [i, j])]) for i in range(4) for j in range(i + 1, 4))
print("s_2 = sum of principal 2x2 minors =", round(s2, 6), "  coefficient of lambda^2:", round(c[2], 6))

pA = 2 * np.linalg.matrix_power(A, 3) - 3 * A @ A + 4 * A - 5 * np.eye(4)
p = lambda x: 2 * x**3 - 3 * x**2 + 4 * x - 5
print("p(A) v = p(lambda) v:", all(np.allclose(pA @ evecs[:, i], p(evals[i]) * evecs[:, i]) for i in range(4)))


eigenvalues: [ 4.   2.5  0.5 -1. ]
A v = lambda v for all four: True
eigenvectors independent (det of the matrix of eigenvectors): -0.4021
char. poly coefficients: [ 1.   -6.    6.25  8.25 -5.  ]
tr A = 6.0 = sum of eigenvalues 6.0
det A = -5.0 = product of eigenvalues -5.0
s_2 = sum of principal 2x2 minors = 6.25   coefficient of lambda^2: 6.25
p(A) v = p(lambda) v: True
